In [1]:
pip install cohere

Note: you may need to restart the kernel to use updated packages.


DEPRECATION: Loading egg at c:\users\legion\appdata\local\programs\python\python312\lib\site-packages\transportation_assignment_lib-0.1-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
import cohere

# Initialize Cohere API
API_KEY = "pYUL24Qf3UJPCqForAQWP8ABzh2wKJzKjhm0pfH4"  
co = cohere.Client(API_KEY)

# Function for EDA
def perform_eda(data):
    eda_results = {
        "Shape": data.shape,
        "Columns": list(data.columns),
        "Missing Values": data.isnull().sum().to_dict(),
        "Data Types": data.dtypes.to_dict(),
        "Correlation Matrix": data.corr().to_dict()
    }
    return eda_results

# Function to build and train a model
def build_model(data, target_col):
    X = data.drop(columns=[target_col])
    y = data[target_col]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    if y.nunique() > 10:  # Regression task
        model = LinearRegression()
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        performance = {
            "MSE": mean_squared_error(y_test, predictions),
            "R2 Score": r2_score(y_test, predictions)
        }
    else:  # Classification task
        model = LogisticRegression()
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        performance = {
            "Accuracy": accuracy_score(y_test, predictions)
        }
    return model, performance

# Function to generate natural language explanations using Cohere
def generate_explanation(text):
    try:
        response = co.generate(
            model="command-xlarge-nightly",
            prompt=text,
            max_tokens=300,
            temperature=0.7
        )
        return response.generations[0].text.strip()
    except Exception as e:
        return f"Error generating explanation: {e}"

# Tkinter UI Functions
def upload_file():
    filepath = filedialog.askopenfilename(filetypes=[("CSV Files", "*.csv")])
    if filepath:
        try:
            data = pd.read_csv(filepath)
            eda_button.config(state="normal")
            model_button.config(state="normal")
            global dataset
            dataset = data
            messagebox.showinfo("File Uploaded", f"Dataset Loaded: {filepath}")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to load file: {e}")

def show_eda():
    if dataset is not None:
        eda_results = perform_eda(dataset)
        eda_text = "\n".join([f"{key}: {value}" for key, value in eda_results.items()])
        natural_language_eda = generate_explanation(
            f"Perform EDA on the following data:\n{eda_text}"
        )
        result_box.delete(1.0, tk.END)
        result_box.insert(tk.END, f"EDA Results:\n{eda_text}\n\nExplanation:\n{natural_language_eda}")
    else:
        messagebox.showerror("Error", "No dataset loaded!")

def build_and_show_model():
    if dataset is not None:
        target_col = target_entry.get()
        if target_col in dataset.columns:
            model, performance = build_model(dataset, target_col)
            performance_text = "\n".join([f"{key}: {value}" for key, value in performance.items()])
            natural_language_performance = generate_explanation(
                f"Explain the performance of the following model:\n{performance_text}"
            )
            result_box.delete(1.0, tk.END)
            result_box.insert(
                tk.END,
                f"Model Performance:\n{performance_text}\n\nExplanation:\n{natural_language_performance}"
            )
        else:
            messagebox.showerror("Error", "Invalid target column name!")
    else:
        messagebox.showerror("Error", "No dataset loaded!")

# Main Tkinter UI
root = tk.Tk()
root.title("LLM-Based EDA and ML Model Builder")
root.geometry("600x600")

# Upload button
upload_button = tk.Button(root, text="Upload Dataset", command=upload_file)
upload_button.pack(pady=10)

# EDA Button
eda_button = tk.Button(root, text="Perform EDA", command=show_eda, state="disabled")
eda_button.pack(pady=10)

# Target Column Entry
tk.Label(root, text="Target Column:").pack(pady=5)
target_entry = tk.Entry(root)
target_entry.pack(pady=5)

# Build Model Button
model_button = tk.Button(root, text="Build Model", command=build_and_show_model, state="disabled")
model_button.pack(pady=10)

# Result Box
result_box = tk.Text(root, wrap="word", height=20)
result_box.pack(pady=10)

root.mainloop()

Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\LEGION\AppData\Local\Programs\Python\Python312\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\LEGION\AppData\Local\Temp\ipykernel_16728\353412234.py", line 90, in build_and_show_model
    model, performance = build_model(dataset, target_col)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LEGION\AppData\Local\Temp\ipykernel_16728\353412234.py", line 40, in build_model
    model.fit(X_train, y_train)
  File "C:\Users\LEGION\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LEGION\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1223, in fit
    X, y = self._validate_data(
           ^^^^^^^^^^^^